## 2. Environment Preparation

Install Unsloth and updated HuggingFace libraries for Llama 3.1 support.

## Step 1: Configuration

All paths and variables for easy configuration.

In [ ]:
# ============================================================================
# CONFIGURATION - All variables for easy setup
# ============================================================================

# Base model configuration
BASE_LLM = "unsloth/Meta-Llama-3.1-8B-Instruct"
MODEL_NAME_BASE = "stoic_llama31_8b_instruct_unsloth"

# Input data configuration
INPUT_DATA_PATH = "/home/spark/projects/augmentoolkit/outputs/marcus_aurelius_dataset"

# Output directory structure - all under ./output/{MODEL_NAME_BASE}/
OUTPUT_BASE_DIR = f"./output/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/adapters"
OUTPUT_DIR_MERGED = f"{OUTPUT_BASE_DIR}/merged"
OUTPUT_DIR_GGUF = f"{OUTPUT_BASE_DIR}/gguf"

# Training configuration
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 5e-5
TARGET_EPOCHS = 1

# LoRA configuration
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# GGUF conversion configuration
LLAMA_CPP_PATH = "/home/spark/resources/llama.cpp"
QUANTIZATION_TYPE = "q8"  # Options: None, "q4", "q8", "q4_k", "q5_k", "q6_k", "fp16"

print("✓ Configuration loaded")
print(f"  Base model: {BASE_LLM}")
print(f"  Model name: {MODEL_NAME_BASE}")
print(f"  Input data: {INPUT_DATA_PATH}")
print(f"  Output base: {OUTPUT_BASE_DIR}")

In [ ]:
# Install core packages from PyPI (much faster than git installs)
!pip install -q unsloth transformers trl peft accelerate datasets bitsandbytes

# Verify installations
import unsloth
import transformers
import trl
print(f"✓ Unsloth: {unsloth.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ TRL: {trl.__version__}")
print("Environment ready!")

## 3. Load Dataset & Format for Instruction Tuning

Load the Augmentoolkit-generated Marcus Aurelius dataset (first-person Stoic responses from Meditations).

In [ ]:
from datasets import load_dataset, concatenate_datasets
import glob

# Load ALL subdirectories and ALL files
all_dirs = glob.glob(f"{INPUT_DATA_PATH}/*/")

print("📚 LOADING ALL AUGMENTOOLKIT DATA")
print(f"Found {len(all_dirs)} subdirectories")

datasets = []
for dir_path in sorted(all_dirs):
    jsonl_files = glob.glob(f"{dir_path}/*.jsonl")
    for file_path in jsonl_files:
        try:
            ds = load_dataset("json", data_files=file_path, split="train")
            datasets.append(ds)
            print(f"  Loaded {len(ds)} from {dir_path.split('/')[-2]}/{file_path.split('/')[-1]}")
        except Exception as e:
            print(f"  Skipped {file_path.split('/')[-1]}: {e}")

dataset = concatenate_datasets(datasets)
dataset = dataset.shuffle(seed=42)

print(f"\n✓ Total: {len(dataset)} examples from ALL Augmentoolkit output")
print(f"✓ Columns: {dataset.column_names}")

import json
print("\n--- Sample ---")
print(json.dumps(dataset[0], indent=2)[:500])

## 4. Load Model & Tokenizer with Unsloth

Load Llama 3.1 8B **Instruct** model for system prompt flexibility - allows persona-switching between different Stoic philosophers.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load model in FULL 16-bit precision (no quantization)
# Uses default HuggingFace cache: ~/.cache/huggingface/hub/ (shared across notebooks)
# INSTRUCT version allows system prompt flexibility for persona-switching
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,  # Full 16-bit precision
    load_in_4bit=False,   # NO quantization
    device_map={"": 0}    # Force all on GPU 0
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✓ Model loaded: {BASE_LLM}")
print(f"✓ Precision: FULL 16-bit (fp16)")
print(f"✓ Tokenizer configured (Instruct model has chat template built-in)")
print(f"✓ Max sequence length: {MAX_SEQ_LENGTH}")
print(f"✓ Using system-wide cache: ~/.cache/huggingface/hub/")

In [ ]:
# Format dataset for Llama 3.1 chat template
# Handle BOTH ShareGPT format (conversations) AND raw text format
def format_instruct(example):
    # If has conversations field AND it's not null, convert ShareGPT to chat template
    if example.get("conversations") is not None:
        messages = []
        for turn in example["conversations"]:
            # ShareGPT uses "from": "system"/"human"/"gpt"
            # Standard uses "role": "system"/"user"/"assistant"
            if turn["from"] == "system":
                messages.append({"role": "system", "content": turn["value"]})
            elif turn["from"] == "human":
                messages.append({"role": "user", "content": turn["value"]})
            elif turn["from"] == "gpt":
                messages.append({"role": "assistant", "content": turn["value"]})
        
        text = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        return {"text": text}
    
    # Otherwise if it has a text field (raw text files), keep it as-is
    elif example.get("text") is not None and len(str(example["text"])) > 0:
        return {"text": str(example["text"])}
    
    # Skip malformed examples
    return {"text": ""}

# Format and keep only text column
dataset = dataset.map(format_instruct, remove_columns=dataset.column_names)

# Filter out empty examples
dataset = dataset.filter(lambda x: len(x["text"]) > 0)

print(f"✓ Dataset formatted: {len(dataset)} examples")
print(f"\n--- Sample formatted text (first 500 chars) ---")
print(dataset[0]['text'][:500])

## 5. Add LoRA Adapters

Configure LoRA for efficient fine-tuning with attention and MLP projection layers.

In [ ]:
from peft import LoraConfig

# Conservative LoRA: lower rank + attention-only for gentle adaptation
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH
)

print(f"✓ LoRA adapters added (r={LORA_R}, targets={LORA_TARGET_MODULES})")
print(f"✓ Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. Trainer Setup & Training

**PURE DOMAIN DATA with LIGHT TRAINING:**
- 100% authentic Stoic examples from Meditations
- 1 epoch with low learning rate to gently teach persona
- This preserves base model capabilities while adding authentic voice

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Calculate training steps
effective_batch_size = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(dataset) // effective_batch_size
max_steps = steps_per_epoch * TARGET_EPOCHS
warmup_steps = max(1, max_steps // 10)
save_steps = steps_per_epoch

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=LEARNING_RATE,
        fp16=True,   # Model loaded in fp16, must train in fp16
        bf16=False,  # Disable bf16 since we're using fp16
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        report_to="none",
        save_strategy="steps",
        save_steps=save_steps,
    )
)

print("✓ Trainer configured for PURE DOMAIN (light training)")
print(f"✓ Dataset size: {len(dataset)} conversations")
print(f"✓ Effective batch size: {effective_batch_size} (batch={BATCH_SIZE} × grad_accum={GRAD_ACCUM})")
print(f"✓ Steps per epoch: {steps_per_epoch}")
print(f"✓ Total epochs: {TARGET_EPOCHS}")
print(f"✓ Total steps: {max_steps}")
print(f"✓ Warmup steps: {warmup_steps}")
print(f"✓ Save every: {save_steps} steps (every epoch)")
print(f"✓ Output directory: {OUTPUT_DIR_ADAPTERS}")

In [ ]:
# Start training
trainer.train()

## 7. Save Model & Inference

Save the fine-tuned model and test inference with a Stoic question.

In [ ]:
# Save merged model
model.save_pretrained_merged(OUTPUT_DIR_MERGED, tokenizer, save_method="merged_16bit")

print(f"✓ Model saved to {OUTPUT_DIR_MERGED}")

In [ ]:
# Prepare model for inference
FastLanguageModel.for_inference(model)

# Test inference with a Stoic question
test_prompt = "What troubles me today is the judgment of others. How should I view this?"

inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": test_prompt}],
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7, 
    top_p=0.9,
    repetition_penalty=1.1
)
response = tokenizer.decode(outputs[0], skip_special_tokens=False)

print("\n=== RAW FULL OUTPUT (with tags) ===")
print(response)
print("\n=== PARSED OUTPUT ===")
print(f"User: {test_prompt}")

# Extract just the assistant response (after the last header)
# Llama 3.1 format: <|start_header_id|>assistant<|end_header_id|>\n\nRESPONSE<|eot_id|>
if "<|start_header_id|>assistant<|end_header_id|>" in response:
    assistant_response = response.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
    assistant_response = assistant_response.replace("<|eot_id|>", "").strip()
    print(f"\nAssistant: {assistant_response}")
else:
    # Fallback for cleaner display
    print(f"\nAssistant: {tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)}")

## Notes

### Configuration
All paths and variables are configured in Step 1 for easy modification:
- `BASE_LLM`: Base model to fine-tune
- `MODEL_NAME_BASE`: Name used for all output folders
- `INPUT_DATA_PATH`: Source data from Augmentoolkit
- Output structure: `./output/{MODEL_NAME_BASE}/{adapters|merged|gguf}`

### Dataset Quality
This notebook uses Augmentoolkit-generated data from Marcus Aurelius' Meditations. The pipeline enforced first-person responses through configuration:
- `shared_instruction`: "You ARE a Stoic philosopher - not explaining Stoicism, but LIVING it."
- Source text: Only Meditations (pure first-person journal)
- All prompts rewritten to enforce "I am a Stoic philosopher..." voice

### Next Steps
- For Epictetus dataset: Run Augmentoolkit with Enchiridion/Discourses
- Merge datasets: Combine Marcus + Epictetus for broader Stoic knowledge
- Evaluate first-person quality: Test if model says "I practice..." vs "Stoics believe..."

## 8. Convert to GGUF for Ollama

Convert the fine-tuned model to GGUF format for use with Ollama. Supports both full-precision (fp32) and quantized formats (q4, q8, etc.) for different size/quality trade-offs.

**Quantization Options:**
- `None`: Full precision (fp32) - maximum quality, largest size
- `"q4"`: 4-bit (Q4_0) - highest compression, fastest inference
- `"q8"`: 8-bit (Q8_0) - best balance of size/speed/quality ⭐ **Recommended**
- `"q4_k"`: 4-bit K-quant (Q4_K_M) - better quality than q4
- `"q5_k"`: 5-bit K-quant (Q5_K_M) - excellent quality, moderate size
- `"q6_k"`: 6-bit K-quant (Q6_K) - near-lossless quality

In [ ]:
from pathlib import Path
import subprocess
import sys

# Create output directory based on quantization type
quant_suffix = "fp32" if QUANTIZATION_TYPE is None else QUANTIZATION_TYPE
OUTPUT_DIR_GGUF_FULL = f"{OUTPUT_DIR_GGUF}/{quant_suffix}"
Path(OUTPUT_DIR_GGUF_FULL).mkdir(parents=True, exist_ok=True)

print(f"🔧 GGUF Conversion for Ollama")
print(f"   Source model: {OUTPUT_DIR_MERGED}")
print(f"   Quantization: {QUANTIZATION_TYPE or 'Full Precision (fp32)'}")
print(f"   Output: {OUTPUT_DIR_GGUF_FULL}")

# Verify llama.cpp exists
if not Path(LLAMA_CPP_PATH).exists():
    raise FileNotFoundError(
        f"❌ llama.cpp not found at {LLAMA_CPP_PATH}\n"
        f"   This DGX uses a shared resources folder.\n"
        f"   Please clone it: git clone https://github.com/ggerganov/llama.cpp {LLAMA_CPP_PATH}\n"
        f"   Then build it: cd {LLAMA_CPP_PATH} && cmake -B build -DLLAMA_CURL=OFF && cmake --build build -j$(nproc)"
    )

print(f"✓ llama.cpp found at {LLAMA_CPP_PATH}")

In [ ]:
from pathlib import Path

if QUANTIZATION_TYPE is None:
    # ========================================================================
    # Full Precision (fp32) - No Quantization
    # ========================================================================
    print("\n[Step 1/1] Converting to full-precision GGUF (fp32)...")
    
    FINAL_GGUF = Path(OUTPUT_DIR_GGUF_FULL) / f"{MODEL_NAME_BASE}-fp32.gguf"
    
    subprocess.run([
        sys.executable,
        str(Path(LLAMA_CPP_PATH) / "convert_hf_to_gguf.py"),
        OUTPUT_DIR_MERGED,
        "--outfile",
        str(FINAL_GGUF),
        "--outtype",
        "f32",  # Full 32-bit precision
    ], check=True)
    
    print(f"   ✅ Full-precision GGUF: {FINAL_GGUF.name}")
    
else:
    # ========================================================================
    # Quantized Conversion (2-step process)
    # ========================================================================
    print("\n[Step 1/2] Converting to fp16 GGUF (pre-quantization)...")
    
    TEMP_GGUF = Path(OUTPUT_DIR_GGUF_FULL) / "temp-fp16.gguf"
    
    subprocess.run([
        sys.executable,
        str(Path(LLAMA_CPP_PATH) / "convert_hf_to_gguf.py"),
        OUTPUT_DIR_MERGED,
        "--outfile",
        str(TEMP_GGUF),
        "--outtype",
        "f16",  # 16-bit precision (required for quantization)
    ], check=True)
    
    print(f"   ✅ fp16 GGUF created: {TEMP_GGUF.name}")
    
    # Step 2: Quantize the fp16 GGUF
    print(f"\n[Step 2/2] Quantizing to {QUANTIZATION_TYPE}...")
    
    FINAL_GGUF = Path(OUTPUT_DIR_GGUF_FULL) / f"{MODEL_NAME_BASE}-{QUANTIZATION_TYPE}.gguf"
    
    # Map friendly names to llama.cpp quantization types
    quant_map = {
        "q4": "Q4_0",
        "q8": "Q8_0",
        "q4_k": "Q4_K_M",
        "q5_k": "Q5_K_M",
        "q6_k": "Q6_K",
        "fp16": "F16",
    }
    llama_quant_type = quant_map.get(QUANTIZATION_TYPE, QUANTIZATION_TYPE.upper())
    
    # Use llama-quantize binary from CMake build
    quantize_binary = Path(LLAMA_CPP_PATH) / "build" / "bin" / "llama-quantize"
    
    if not quantize_binary.exists():
        raise FileNotFoundError(
            f"❌ llama-quantize binary not found at {quantize_binary}\n"
            f"   Please build llama.cpp:\n"
            f"   cd {LLAMA_CPP_PATH} && cmake -B build -DLLAMA_CURL=OFF && cmake --build build --config Release -j$(nproc)"
        )
    
    subprocess.run([
        str(quantize_binary),
        str(TEMP_GGUF),
        str(FINAL_GGUF),
        llama_quant_type,
    ], check=True)
    
    # Clean up temp file
    TEMP_GGUF.unlink()
    print(f"   ✅ Quantized GGUF: {FINAL_GGUF.name} ({llama_quant_type})")

print(f"\n✅ GGUF conversion complete: {FINAL_GGUF}")

In [ ]:
# Create Modelfile for Ollama
print("\n📝 Creating Modelfile for Open WebUI...")

MODELFILE_PATH = Path(OUTPUT_DIR_GGUF_FULL) / "Modelfile"

# Minimal Modelfile for Open WebUI (it manages system prompts and parameters via UI)
# Llama 3.1 chat template with proper special tokens
modelfile_content = f"""FROM ./{FINAL_GGUF.name}

TEMPLATE \"\"\"{{{{ if .System }}}}<|start_header_id|>system<|end_header_id|>

{{{{ .System }}}}<|eot_id|>{{{{ end }}}}{{{{ if .Prompt }}}}<|start_header_id|>user<|end_header_id|>

{{{{ .Prompt }}}}<|eot_id|>{{{{ end }}}}<|start_header_id|>assistant<|end_header_id|>

\"\"\"

PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|end_of_text|>"
"""

MODELFILE_PATH.write_text(modelfile_content, encoding="utf-8")

print(f"✅ Modelfile created: {MODELFILE_PATH}")
print(f"\n🚀 Ready for Open WebUI!")
print(f"\nTo import into Ollama:")
print(f"   cd {OUTPUT_DIR_GGUF_FULL}")
print(f"   ollama create {MODEL_NAME_BASE} -f Modelfile")
print(f"\nThen configure in Open WebUI:")
print(f"   - Select '{MODEL_NAME_BASE}' as base model")
print(f"   - Add custom system prompt")
print(f"   - Set temperature, top_p, etc. via UI sliders")